# Núcleos de población: comprobar y preparar (paso 2 para los orígenes)

**Para qué:** los núcleos son los **orígenes** de todas las preguntas (desde dónde se mide el acceso),
llevan la **población** (para ponderar y filtrar) y sus **nombres** son la base de `resolve_place`.

**Fuente:** capa de *localidades* del SIG de Asturias (servicio `UnidadesAdministrativas`), descargada con
`download_sig_asturias.ipynb`. Según la descripción del servicio trae, por punto, el **código INE de 11 dígitos**,
el nombre, el tipo, el rango (capital, localidad principal…) y los **habitantes (fuente INE)**.
Si es así, ya no hace falta cruzar SADEI con IGN: puntos y población vienen juntos.

**Qué comprueba este cuaderno** (antes de fiarnos de la capa):
1. Cuántos puntos hay y de qué tipo (¿entidades singulares, núcleos, todo?).
2. Si el código INE es único y si su municipio coincide con el campo de municipio.
3. Población total y cuántos puntos tienen 0 habitantes. Referencias: SADEI 2025 da **6.983 entidades
   singulares, 815 sin residentes**; Asturias tiene alrededor de un millón de habitantes.
4. **Topónimos repetidos**: cuántos nombres aparecen en varios sitios (es la hipótesis de las preguntas ambiguas).
5. Cruce espacial con las **parroquias** y con las **zonas básicas de salud**.

Y guarda la tabla `settlements` en `data/processed/asturias.gpkg` con un esquema común.

In [ ]:
%pip install -q geopandas

In [ ]:
import glob
import os
import re
import unicodedata

import geopandas as gpd
import pandas as pd

RAW_DIR = "data/raw"
PROCESSED_DIR = "data/processed"
GPKG_PATH = os.path.join(PROCESSED_DIR, "asturias.gpkg")
CRS_METRES = "EPSG:25830"

os.makedirs(PROCESSED_DIR, exist_ok=True)

## 1. Encontrar y leer las capas

No sabemos de antemano el nombre exacto de cada capa en el servidor, así que se buscan por sus **columnas**:
la de localidades es la que tiene `codigoine` y `habitantes`. Si no aparece, mira la lista que imprime la celda.

In [ ]:
def read_layer(path):
    """Read a GeoJSON layer with lower-case column names."""
    gdf = gpd.read_file(path)
    gdf.columns = [c.lower() if c != "geometry" else c for c in gdf.columns]
    return gdf


def find_layer(required_columns):
    """Return the newest raw layer that has all the required columns, or None."""
    for path in sorted(glob.glob(os.path.join(RAW_DIR, "*.geojson")), reverse=True):
        columns = {c.lower() for c in gpd.read_file(path, rows=1).columns}
        if required_columns <= columns:
            return path
    return None


for path in sorted(glob.glob(os.path.join(RAW_DIR, "unidadesadministrativas__*.geojson"))
                   + glob.glob(os.path.join(RAW_DIR, "salud__*.geojson"))):
    print(os.path.basename(path))

LOCALITIES_PATH = find_layer({"codigoine", "habitantes"})
PARISHES_PATH = find_layer({"code", "topo_ofi", "concejo"})
HEALTH_ZONES_PATH = find_layer({"zona_1", "cabecera", "c_salud"})
print("\nlocalidades:", LOCALITIES_PATH)
print("parroquias:", PARISHES_PATH)
print("zonas de salud:", HEALTH_ZONES_PATH)

In [ ]:
localities = read_layer(LOCALITIES_PATH).to_crs(CRS_METRES)
print(len(localities), "puntos |", localities.geom_type.unique().tolist())
print("columnas:", [c for c in localities.columns if c != "geometry"])
display(localities.drop(columns="geometry").head())

## 2. Tipos de punto y código INE

El código INE de una unidad poblacional tiene 11 dígitos: **provincia (2) + municipio (3) + entidad colectiva (2)
+ entidad singular (2) + núcleo/diseminado (2)**. Asturias es la provincia 33.
Si los dos últimos dígitos son siempre `00`, los puntos son entidades singulares; si no, hay núcleos.

Problemas vistos en la descarga del 11/10/2026: códigos vacíos, códigos con un salto de línea al final,
un código de 7 dígitos y unos pocos códigos repetidos en puntos distintos (a kilómetros, así que no son
duplicados del mismo sitio). El municipio se toma de `inemuni`, no del código: en los dos casos en que
no coinciden, `inemuni` y `concejo` dicen lo mismo y el código es el erróneo.

In [ ]:
codes = localities["codigoine"].astype(str).str.strip()
valid_code = codes.str.fullmatch(r"33\d{9}")
print("códigos válidos (11 dígitos, provincia 33):", int(valid_code.sum()), "de", len(codes))
print("códigos no válidos:", localities.loc[~valid_code, ["codigoine", "nombre", "concejo"]].to_dict("records"))
print("códigos repetidos:", int(codes[valid_code].duplicated(keep=False).sum()), "puntos")
municipality_codes = localities["inemuni"].astype(str).str.strip().str.zfill(5)
print("provincias:", codes.str[:2].value_counts().to_dict())
print("últimos 2 dígitos (núcleo/diseminado), los 5 más frecuentes:",
      codes.str[-2:].value_counts().head().to_dict())

for column in ["tipo", "rango"]:
    if column in localities:
        print(f"\n{column}:")
        print(localities[column].value_counts(dropna=False).to_string())

# Does the municipality in the code match the municipality field?
mismatch = valid_code & (codes.str[:5] != municipality_codes)
print("\ncódigo INE con municipio distinto de 'inemuni':")
print(localities.loc[mismatch, ["codigoine", "nombre", "inemuni", "concejo"]].to_string())

# Unique id: the INE code when it is valid and unique; otherwise municipality + row number.
unique_code = valid_code & ~codes.duplicated(keep=False)
settlement_ids = codes.where(unique_code, municipality_codes + "-x" + localities.index.astype(str))
assert settlement_ids.is_unique
print("puntos con id provisional (código vacío, mal formado o repetido):", int((~unique_code).sum()))

## 3. Población

Compara con las referencias de arriba. Si la suma se aleja mucho de un millón, la capa no tiene todas las
unidades o mezcla niveles (por ejemplo, entidades y núcleos a la vez, contando dos veces a la misma gente).

In [ ]:
population = pd.to_numeric(localities["habitantes"], errors="coerce")
print("población total:", f"{population.sum():,.0f}")
print("sin dato de población:", int(population.isna().sum()))
print("con 0 habitantes:", int((population == 0).sum()))
print("\npercentiles de población por punto:")
print(population.describe(percentiles=[.25, .5, .75, .9, .99]).round(0).to_string())

## 4. Topónimos repetidos

Se normaliza el nombre (minúsculas, sin tildes, sin artículos al final tipo "Vega (La)") y se cuenta cuántas
veces aparece cada uno. **Esto decide si las preguntas de "topónimo ambiguo" tienen sentido en el TFG.**

Unos 100 nombres son bilingües, con barra: `Cuadonga/Covadonga`, `Cangues d'Onís/Cangas de Onís`. Se guardan
las dos formas por separado (`name_alt`) para que el agente encuentre el núcleo escriba la que escriba.

In [ ]:
def normalize_name(name):
    """Lower-case, strip accents and move trailing articles: 'Vega (La)' -> 'la vega'."""
    text = str(name).strip()
    match = re.match(r"^(.*)\s*\((el|la|los|las|l'|les)\)$", text, flags=re.I)
    if match:
        text = f"{match.group(2)} {match.group(1)}"
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    return re.sub(r"\s+", " ", text.lower()).strip()


name_parts = localities["nombre"].astype(str).str.split("/", n=1, expand=True).reindex(columns=[0, 1])
localities["name_alt"] = name_parts[1].str.strip()
localities["name_normalized"] = name_parts[0].map(normalize_name)
print("nombres con barra (bilingües):", int(localities["name_alt"].notna().sum()))
localities["municipality_code"] = municipality_codes
counts = localities.groupby("name_normalized").agg(
    places=("name_normalized", "size"),
    municipalities=("municipality_code", "nunique"),
)
repeated = counts[counts["places"] > 1].sort_values("places", ascending=False)
print("nombres distintos:", len(counts))
print("nombres que se repiten:", len(repeated),
      f"({repeated['places'].sum()} puntos, {repeated['places'].sum() / len(localities):.0%} del total)")
print("repetidos en más de un concejo:", int((repeated["municipalities"] > 1).sum()))
display(repeated.head(20))

## 5. Cruce con parroquias y zonas básicas de salud

Cada punto se asigna a la parroquia y a la zona de salud en la que cae (`sjoin` con `within`).
Los puntos que no caen en ninguna (en la costa, por ejemplo) se buscan con el polígono más cercano.

In [ ]:
def attach_polygon(points, polygons_path, columns, prefix):
    """Add the attributes of the polygon containing each point (nearest polygon if none)."""
    polygons = read_layer(polygons_path).to_crs(CRS_METRES)[columns + ["geometry"]]
    polygons = polygons.rename(columns={c: f"{prefix}{c}" for c in columns})
    joined = gpd.sjoin(points, polygons, how="left", predicate="within").drop(columns="index_right")
    joined = joined[~joined.index.duplicated()]  # a point on a shared border falls in two polygons
    missing = joined[f"{prefix}{columns[0]}"].isna()
    if missing.any():
        nearest = gpd.sjoin_nearest(points[missing], polygons, how="left").drop(columns="index_right")
        nearest = nearest[~nearest.index.duplicated()]
        joined.loc[missing, nearest.columns] = nearest
    print(f"{prefix}: {int(missing.sum())} puntos fuera de todo polígono, asignados al más cercano")
    return joined


if PARISHES_PATH:
    localities = attach_polygon(localities, PARISHES_PATH, ["code", "topo_ofi", "concejo"], "parish_")
if HEALTH_ZONES_PATH:
    localities = attach_polygon(localities, HEALTH_ZONES_PATH, ["zona_1", "nombre", "cabecera"], "hz_")

# Consistency check: does each point fall inside the municipality it claims? Names differ between layers
# ("REGUERAS, LAS" vs "Las Regueras"), so compare INE codes against the municipalities layer.
MUNICIPALITIES_PATH = find_layer({"codigo_ine", "conceyu"})
if MUNICIPALITIES_PATH:
    checked = attach_polygon(localities[["municipality_code", "nombre", "geometry"]], MUNICIPALITIES_PATH,
                             ["codigo_ine", "concejo"], "mun_")
    outside = checked[checked["municipality_code"] != checked["mun_codigo_ine"].astype(str)]
    print("puntos que caen en otro concejo distinto al suyo:", len(outside))
    display(outside.drop(columns="geometry").head(10))

## 6. Guardar con el esquema común

Columnas en inglés, como el resto del código. Se guardan todos los puntos, también los de 0 habitantes:
no son orígenes útiles, pero sí nombres que el usuario puede escribir.

In [ ]:
settlements = gpd.GeoDataFrame({
    "settlement_id": settlement_ids,
    "ine_code": codes.where(valid_code),
    "name": localities["nombre"],
    "name_alt": localities["name_alt"],
    "name_normalized": localities["name_normalized"],
    "municipality_code": municipality_codes,
    "parish_code": localities.get("parish_code"),
    "parish": localities.get("parish_topo_ofi"),
    "health_zone": localities.get("hz_zona_1"),
    "population": population,
    "rank": localities.get("rango"),
    "type": localities.get("tipo"),
}, geometry=localities.geometry, crs=CRS_METRES)

settlements.to_file(GPKG_PATH, layer="settlements", driver="GPKG")
print(f"{len(settlements)} núcleos → {GPKG_PATH} (capa 'settlements')")
display(settlements.drop(columns="geometry").head())

## Qué mirar y contarme

1. **Número de puntos y población total** frente a las referencias (6.983 entidades, ~1 millón de habitantes).
2. **Qué son los puntos**: el reparto de `tipo`/`rango` y los dos últimos dígitos del código.
3. **Topónimos repetidos**: cuántos y los 20 primeros. Si son muchos, las preguntas ambiguas están justificadas con datos.
4. Puntos fuera de parroquia o en otro concejo: si son pocos, normal (costa, bordes).

**Ojo con la población en ciudades:** la capital de cada concejo lleva toda la población de su entidad
(Gijón-centro, 253.423). Sumar población por zona de salud funciona en zona rural, pero en Oviedo, Gijón
o Avilés da cifras absurdas (zonas urbanas con 172 habitantes). Para el TFG da igual: los orígenes que
interesan son rurales.

Si la capa no trae habitantes o no tiene todas las entidades, el plan B sigue siendo el nomenclátor de SADEI.